# 02 - Feature Engineering

Tao du lieu dau vao cho bai toan du bao `traffic_volume`.

## 1. Doc du lieu goc

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

In [2]:
DATA_FILE = 'Metro_Interstate_Traffic_Volume.csv'

candidate_paths = [
    Path.cwd() / '..' / 'data' / 'raw' / DATA_FILE,
    Path.cwd() / 'data' / 'raw' / DATA_FILE,
    Path.cwd() / 'time_series _' / 'data' / 'raw' / DATA_FILE,
]

raw_path = next((path.resolve() for path in candidate_paths if path.exists()), None)
if raw_path is None:
    raise FileNotFoundError('Khong tim thay file raw data.')

project_dir = raw_path.parents[2]
processed_dir = project_dir / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(raw_path)
print(f'Raw path: {raw_path}')
print(f'Processed dir: {processed_dir}')
print(f'Shape raw: {df_raw.shape}')
display(df_raw.head())

Raw path: D:\Code\Python\time_series _\data\raw\Metro_Interstate_Traffic_Volume.csv
Processed dir: D:\Code\Python\time_series _\data\processed
Shape raw: (48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.2800,0.0000,0.0000,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.3600,0.0000,0.0000,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.5800,0.0000,0.0000,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.1300,0.0000,0.0000,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.1400,0.0000,0.0000,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


## 2. Chuyen `date_time` sang datetime

In [3]:
df = df_raw.copy()
df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')

print(df['date_time'].dtype)
print(f"So dong date_time khong parse duoc: {df['date_time'].isna().sum():,}")
df = df.dropna(subset=['date_time']).copy()
display(df[['date_time']].head())

datetime64[us]
So dong date_time khong parse duoc: 0


,date_time
0,2012-10-02 09:00:00
1,2012-10-02 10:00:00
2,2012-10-02 11:00:00
3,2012-10-02 12:00:00
4,2012-10-02 13:00:00


## 3. Sap xep du lieu theo thoi gian

In [4]:
df = df.sort_values('date_time').reset_index(drop=True)

print(f"Tu ngay: {df['date_time'].min()}")
print(f"Den ngay: {df['date_time'].max()}")
print(f"Da sap xep tang dan: {df['date_time'].is_monotonic_increasing}")

Tu ngay: 2012-10-02 09:00:00
Den ngay: 2018-09-30 23:00:00
Da sap xep tang dan: True


## 4. Gop cac dong trung `date_time`

In [5]:
def mode_or_first(series):
    mode_values = series.dropna().mode()
    if len(mode_values) > 0:
        return mode_values.iloc[0]
    return np.nan

numeric_agg = {
    'temp': 'mean',
    'rain_1h': 'mean',
    'snow_1h': 'mean',
    'clouds_all': 'mean',
    'traffic_volume': 'mean',
}
categorical_agg = {
    'holiday': mode_or_first,
    'weather_main': mode_or_first,
    'weather_description': mode_or_first,
}

duplicate_count = df.duplicated('date_time').sum()
df_hourly_base = (
    df.groupby('date_time', as_index=False)
    .agg({**numeric_agg, **categorical_agg})
    .sort_values('date_time')
    .reset_index(drop=True)
)

print(f'So dong trung date_time truoc khi gop: {duplicate_count:,}')
print(f'Shape sau khi gop: {df_hourly_base.shape}')
display(df_hourly_base.head())

So dong trung date_time truoc khi gop: 7,629
Shape sau khi gop: (40575, 9)


,date_time,temp,rain_1h,snow_1h,clouds_all,traffic_volume,holiday,weather_main,weather_description
0,2012-10-02 09:00:00,288.2800,0.0000,0.0000,40.0000,"5,545.0000",NaN,Clouds,scattered clouds
1,2012-10-02 10:00:00,289.3600,0.0000,0.0000,75.0000,"4,516.0000",NaN,Clouds,broken clouds
2,2012-10-02 11:00:00,289.5800,0.0000,0.0000,90.0000,"4,767.0000",NaN,Clouds,overcast clouds
3,2012-10-02 12:00:00,290.1300,0.0000,0.0000,90.0000,"5,026.0000",NaN,Clouds,overcast clouds
4,2012-10-02 13:00:00,291.1400,0.0000,0.0000,75.0000,"4,918.0000",NaN,Clouds,broken clouds


## 5. Resample du lieu theo tung gio

In [6]:
df_hourly = df_hourly_base.set_index('date_time').asfreq('h')
df_hourly = df_hourly.reset_index()

print(f'Shape sau khi resample hourly: {df_hourly.shape}')
print(f"So moc gio bi thieu duoc tao them: {df_hourly['traffic_volume'].isna().sum():,}")
display(df_hourly.head())

Shape sau khi resample hourly: (52551, 9)
So moc gio bi thieu duoc tao them: 11,976


,date_time,temp,rain_1h,snow_1h,clouds_all,traffic_volume,holiday,weather_main,weather_description
0,2012-10-02 09:00:00,288.2800,0.0000,0.0000,40.0000,"5,545.0000",NaN,Clouds,scattered clouds
1,2012-10-02 10:00:00,289.3600,0.0000,0.0000,75.0000,"4,516.0000",NaN,Clouds,broken clouds
2,2012-10-02 11:00:00,289.5800,0.0000,0.0000,90.0000,"4,767.0000",NaN,Clouds,overcast clouds
3,2012-10-02 12:00:00,290.1300,0.0000,0.0000,90.0000,"5,026.0000",NaN,Clouds,overcast clouds
4,2012-10-02 13:00:00,291.1400,0.0000,0.0000,75.0000,"4,918.0000",NaN,Clouds,broken clouds


## 6. Xu ly missing value

In [7]:
missing_before = df_hourly.isna().sum().to_frame('missing_before')
display(missing_before)

,missing_before
date_time,0
temp,11976
rain_1h,11976
snow_1h,11976
clouds_all,11976
traffic_volume,11976
holiday,52498
weather_main,11976
weather_description,11976


In [8]:
numeric_cols = ['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'traffic_volume']
categorical_cols = ['holiday', 'weather_main', 'weather_description']

df_hourly[numeric_cols] = df_hourly[numeric_cols].interpolate(method='linear', limit_direction='both')
df_hourly[categorical_cols] = df_hourly[categorical_cols].ffill().bfill().fillna('Unknown')
df_hourly['holiday'] = df_hourly['holiday'].fillna('None')

missing_after = df_hourly.isna().sum().to_frame('missing_after')
display(missing_before.join(missing_after))

,missing_before,missing_after
date_time,0,0
temp,11976,0
rain_1h,11976,0
snow_1h,11976,0
clouds_all,11976,0
traffic_volume,11976,0
holiday,52498,0
weather_main,11976,0
weather_description,11976,0


## 7. Tao dac trung thoi gian

In [9]:
features_df = df_hourly.copy()

features_df['hour'] = features_df['date_time'].dt.hour
features_df['day_of_week'] = features_df['date_time'].dt.dayofweek
features_df['day_of_month'] = features_df['date_time'].dt.day
features_df['day_of_year'] = features_df['date_time'].dt.dayofyear
features_df['week_of_year'] = features_df['date_time'].dt.isocalendar().week.astype(int)
features_df['month'] = features_df['date_time'].dt.month
features_df['quarter'] = features_df['date_time'].dt.quarter
features_df['year'] = features_df['date_time'].dt.year

display(features_df[['date_time', 'hour', 'day_of_week', 'day_of_month', 'day_of_year', 'week_of_year', 'month', 'quarter', 'year']].head())

,date_time,hour,day_of_week,day_of_month,day_of_year,week_of_year,month,quarter,year
0,2012-10-02 09:00:00,9,1,2,276,40,10,4,2012
1,2012-10-02 10:00:00,10,1,2,276,40,10,4,2012
2,2012-10-02 11:00:00,11,1,2,276,40,10,4,2012
3,2012-10-02 12:00:00,12,1,2,276,40,10,4,2012
4,2012-10-02 13:00:00,13,1,2,276,40,10,4,2012


## 8. Tao dac trung chu ky bang sin/cos

In [10]:
features_df['hour_sin'] = np.sin(2 * np.pi * features_df['hour'] / 24)
features_df['hour_cos'] = np.cos(2 * np.pi * features_df['hour'] / 24)
features_df['day_of_week_sin'] = np.sin(2 * np.pi * features_df['day_of_week'] / 7)
features_df['day_of_week_cos'] = np.cos(2 * np.pi * features_df['day_of_week'] / 7)
features_df['month_sin'] = np.sin(2 * np.pi * features_df['month'] / 12)
features_df['month_cos'] = np.cos(2 * np.pi * features_df['month'] / 12)
features_df['day_of_year_sin'] = np.sin(2 * np.pi * features_df['day_of_year'] / 365.25)
features_df['day_of_year_cos'] = np.cos(2 * np.pi * features_df['day_of_year'] / 365.25)

display(features_df[['hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos']].head())

,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos
0,0.7071,-0.7071,0.7818,0.6235,-0.8660,0.5000
1,0.5000,-0.8660,0.7818,0.6235,-0.8660,0.5000
2,0.2588,-0.9659,0.7818,0.6235,-0.8660,0.5000
3,0.0000,-1.0000,0.7818,0.6235,-0.8660,0.5000
4,-0.2588,-0.9659,0.7818,0.6235,-0.8660,0.5000


## 9. Tao bien `is_weekend`

In [11]:
features_df['is_weekend'] = features_df['day_of_week'].isin([5, 6]).astype(int)
features_df[['date_time', 'day_of_week', 'is_weekend']].head(10)

,date_time,day_of_week,is_weekend
0,2012-10-02 09:00:00,1,0
1,2012-10-02 10:00:00,1,0
2,2012-10-02 11:00:00,1,0
3,2012-10-02 12:00:00,1,0
4,2012-10-02 13:00:00,1,0
5,2012-10-02 14:00:00,1,0
6,2012-10-02 15:00:00,1,0
7,2012-10-02 16:00:00,1,0
8,2012-10-02 17:00:00,1,0
9,2012-10-02 18:00:00,1,0


## 10. Tao bien `is_holiday`

In [12]:
features_df['is_holiday'] = (features_df['holiday'].fillna('None') != 'None').astype(int)
display(features_df['is_holiday'].value_counts().to_frame('count'))
display(features_df.loc[features_df['is_holiday'] == 1, ['date_time', 'holiday', 'traffic_volume']].head(20))

,count
is_holiday,
1,52551


,date_time,holiday,traffic_volume
0,2012-10-02 09:00:00,Columbus Day,"5,545.0000"
1,2012-10-02 10:00:00,Columbus Day,"4,516.0000"
2,2012-10-02 11:00:00,Columbus Day,"4,767.0000"
3,2012-10-02 12:00:00,Columbus Day,"5,026.0000"
4,2012-10-02 13:00:00,Columbus Day,"4,918.0000"
5,2012-10-02 14:00:00,Columbus Day,"5,181.0000"
6,2012-10-02 15:00:00,Columbus Day,"5,584.0000"
7,2012-10-02 16:00:00,Columbus Day,"6,015.0000"
8,2012-10-02 17:00:00,Columbus Day,"5,791.0000"
9,2012-10-02 18:00:00,Columbus Day,"4,770.0000"


## 11. One-hot encoding `weather_main`

In [13]:
weather_dummies = pd.get_dummies(features_df['weather_main'], prefix='weather_main', dtype=int)
features_df = pd.concat([features_df, weather_dummies], axis=1)
weather_feature_cols = weather_dummies.columns.tolist()

print(weather_feature_cols)
display(features_df[['weather_main'] + weather_feature_cols].head())

['weather_main_Clear', 'weather_main_Clouds', 'weather_main_Drizzle', 'weather_main_Fog', 'weather_main_Haze', 'weather_main_Mist', 'weather_main_Rain', 'weather_main_Smoke', 'weather_main_Snow', 'weather_main_Squall', 'weather_main_Thunderstorm']


,weather_main,weather_main_Clear,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,weather_main_Rain,weather_main_Smoke,weather_main_Snow,weather_main_Squall,weather_main_Thunderstorm
0,Clouds,0,1,0,0,0,0,0,0,0,0,0
1,Clouds,0,1,0,0,0,0,0,0,0,0,0
2,Clouds,0,1,0,0,0,0,0,0,0,0,0
3,Clouds,0,1,0,0,0,0,0,0,0,0,0
4,Clouds,0,1,0,0,0,0,0,0,0,0,0


## 12. Chon danh sach feature dau vao

In [14]:
base_feature_cols = [
    'temp',
    'rain_1h',
    'snow_1h',
    'clouds_all',
    'hour',
    'day_of_week',
    'day_of_month',
    'day_of_year',
    'week_of_year',
    'month',
    'quarter',
    'year',
    'hour_sin',
    'hour_cos',
    'day_of_week_sin',
    'day_of_week_cos',
    'month_sin',
    'month_cos',
    'day_of_year_sin',
    'day_of_year_cos',
    'is_weekend',
    'is_holiday',
]

feature_cols = base_feature_cols + weather_feature_cols
print(f'So feature dau vao: {len(feature_cols)}')
print(feature_cols)

So feature dau vao: 33
['temp', 'rain_1h', 'snow_1h', 'clouds_all', 'hour', 'day_of_week', 'day_of_month', 'day_of_year', 'week_of_year', 'month', 'quarter', 'year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'is_weekend', 'is_holiday', 'weather_main_Clear', 'weather_main_Clouds', 'weather_main_Drizzle', 'weather_main_Fog', 'weather_main_Haze', 'weather_main_Mist', 'weather_main_Rain', 'weather_main_Smoke', 'weather_main_Snow', 'weather_main_Squall', 'weather_main_Thunderstorm']


## 13. Xac dinh target la `traffic_volume`

In [15]:
target_col = 'traffic_volume'

model_df = features_df[['date_time'] + feature_cols + [target_col]].copy()
model_df[feature_cols + [target_col]] = model_df[feature_cols + [target_col]].astype(float)

print(model_df.shape)
display(model_df.head())

(52551, 35)


,date_time,temp,rain_1h,snow_1h,clouds_all,hour,day_of_week,day_of_month,day_of_year,week_of_year,month,quarter,year,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,day_of_year_sin,day_of_year_cos,is_weekend,is_holiday,weather_main_Clear,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,weather_main_Rain,weather_main_Smoke,weather_main_Snow,weather_main_Squall,weather_main_Thunderstorm,traffic_volume
0,2012-10-02 09:00:00,288.2800,0.0000,0.0000,40.0000,9.0000,1.0000,2.0000,276.0000,40.0000,10.0000,4.0000,"2,012.0000",0.7071,-0.7071,0.7818,0.6235,-0.8660,0.5000,-0.9994,0.0355,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"5,545.0000"
1,2012-10-02 10:00:00,289.3600,0.0000,0.0000,75.0000,10.0000,1.0000,2.0000,276.0000,40.0000,10.0000,4.0000,"2,012.0000",0.5000,-0.8660,0.7818,0.6235,-0.8660,0.5000,-0.9994,0.0355,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"4,516.0000"
2,2012-10-02 11:00:00,289.5800,0.0000,0.0000,90.0000,11.0000,1.0000,2.0000,276.0000,40.0000,10.0000,4.0000,"2,012.0000",0.2588,-0.9659,0.7818,0.6235,-0.8660,0.5000,-0.9994,0.0355,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"4,767.0000"
3,2012-10-02 12:00:00,290.1300,0.0000,0.0000,90.0000,12.0000,1.0000,2.0000,276.0000,40.0000,10.0000,4.0000,"2,012.0000",0.0000,-1.0000,0.7818,0.6235,-0.8660,0.5000,-0.9994,0.0355,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"5,026.0000"
4,2012-10-02 13:00:00,291.1400,0.0000,0.0000,75.0000,13.0000,1.0000,2.0000,276.0000,40.0000,10.0000,4.0000,"2,012.0000",-0.2588,-0.9659,0.7818,0.6235,-0.8660,0.5000,-0.9994,0.0355,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,"4,918.0000"


## 14. Chia train/validation/test theo thoi gian ti le 70/15/15

In [16]:
n = len(model_df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = model_df.iloc[:train_end].copy()
val_df = model_df.iloc[train_end:val_end].copy()
test_df = model_df.iloc[val_end:].copy()

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'start': [train_df['date_time'].min(), val_df['date_time'].min(), test_df['date_time'].min()],
    'end': [train_df['date_time'].max(), val_df['date_time'].max(), test_df['date_time'].max()],
})
display(split_summary)

,split,rows,start,end
0,train,36785,2012-10-02 09:00:00,2016-12-13 01:00:00
1,validation,7883,2016-12-13 02:00:00,2017-11-06 12:00:00
2,test,7883,2017-11-06 13:00:00,2018-09-30 23:00:00


## 15. Chuan hoa du lieu bang `StandardScaler`

In [17]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(train_df[feature_cols])
X_val_scaled = feature_scaler.transform(val_df[feature_cols])
X_test_scaled = feature_scaler.transform(test_df[feature_cols])

y_train_scaled = target_scaler.fit_transform(train_df[[target_col]]).ravel()
y_val_scaled = target_scaler.transform(val_df[[target_col]]).ravel()
y_test_scaled = target_scaler.transform(test_df[[target_col]]).ravel()

print(X_train_scaled.shape, X_val_scaled.shape, X_test_scaled.shape)
print(y_train_scaled.shape, y_val_scaled.shape, y_test_scaled.shape)

(36785, 33) (7883, 33) (7883, 33)
(36785,) (7883,) (7883,)


## 16. Tao window `seq_len = 168`, `pred_len = 24`

In [18]:
seq_len = 168
pred_len = 24

def create_windows(X, y, date_time_values, seq_len=168, pred_len=24):
    X_windows = []
    y_windows = []
    start_times = []
    target_start_times = []
    target_end_times = []
    max_start = len(X) - seq_len - pred_len + 1
    
    for start in range(max(0, max_start)):
        input_end = start + seq_len
        target_end = input_end + pred_len
        X_windows.append(X[start:input_end])
        y_windows.append(y[input_end:target_end])
        start_times.append(date_time_values[start])
        target_start_times.append(date_time_values[input_end])
        target_end_times.append(date_time_values[target_end - 1])
    
    return {
        'X': np.asarray(X_windows, dtype=np.float32),
        'y': np.asarray(y_windows, dtype=np.float32),
        'window_start': np.asarray(start_times, dtype='datetime64[ns]'),
        'target_start': np.asarray(target_start_times, dtype='datetime64[ns]'),
        'target_end': np.asarray(target_end_times, dtype='datetime64[ns]'),
    }

train_windows = create_windows(X_train_scaled, y_train_scaled, train_df['date_time'].to_numpy(), seq_len, pred_len)
val_windows = create_windows(X_val_scaled, y_val_scaled, val_df['date_time'].to_numpy(), seq_len, pred_len)
test_windows = create_windows(X_test_scaled, y_test_scaled, test_df['date_time'].to_numpy(), seq_len, pred_len)

print(f"Train windows X/y: {train_windows['X'].shape} / {train_windows['y'].shape}")
print(f"Val windows X/y: {val_windows['X'].shape} / {val_windows['y'].shape}")
print(f"Test windows X/y: {test_windows['X'].shape} / {test_windows['y'].shape}")

Train windows X/y: (36594, 168, 33) / (36594, 24)
Val windows X/y: (7692, 168, 33) / (7692, 24)
Test windows X/y: (7692, 168, 33) / (7692, 24)


## 17. Luu du lieu da xu ly

In [19]:
model_df.to_csv(processed_dir / 'traffic_features_hourly.csv', index=False)
train_df.to_csv(processed_dir / 'train.csv', index=False)
val_df.to_csv(processed_dir / 'validation.csv', index=False)
test_df.to_csv(processed_dir / 'test.csv', index=False)

np.savez_compressed(processed_dir / 'train_windows.npz', **train_windows)
np.savez_compressed(processed_dir / 'validation_windows.npz', **val_windows)
np.savez_compressed(processed_dir / 'test_windows.npz', **test_windows)

joblib.dump(feature_scaler, processed_dir / 'feature_scaler.pkl')
joblib.dump(target_scaler, processed_dir / 'target_scaler.pkl')

metadata = {
    'raw_path': str(raw_path),
    'processed_dir': str(processed_dir),
    'target_col': target_col,
    'feature_cols': feature_cols,
    'weather_feature_cols': weather_feature_cols,
    'seq_len': seq_len,
    'pred_len': pred_len,
    'split_ratio': {'train': 0.70, 'validation': 0.15, 'test': 0.15},
    'split_rows': {'train': len(train_df), 'validation': len(val_df), 'test': len(test_df)},
    'window_shapes': {
        'train_X': list(train_windows['X'].shape),
        'train_y': list(train_windows['y'].shape),
        'validation_X': list(val_windows['X'].shape),
        'validation_y': list(val_windows['y'].shape),
        'test_X': list(test_windows['X'].shape),
        'test_y': list(test_windows['y'].shape),
    },
}

with open(processed_dir / 'feature_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

with open(processed_dir / 'feature_columns.json', 'w', encoding='utf-8') as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

print('Da luu cac file:')
for path in sorted(processed_dir.iterdir()):
    print(path.name)

Da luu cac file:
feature_columns.json
feature_metadata.json
feature_scaler.pkl
target_scaler.pkl
test.csv
test_windows.npz
traffic_features_hourly.csv
train.csv
train_windows.npz
validation.csv
validation_windows.npz
